# AI-Based Tomato Leaf Disease Detection
## Training Notebook — Google Colab
**Madda Walabu University | College of Computing**

## Step 1: Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install tensorflow numpy pandas matplotlib seaborn opencv-python scikit-learn tqdm -q

## Step 2: Download PlantVillage Dataset from Kaggle

In [ ]:
# Upload your kaggle.json API key first
from google.colab import files
files.upload()  # Upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip -d dataset/
print('Dataset downloaded!')

## Step 3: Imports & Configuration

In [ ]:
import os, json, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import seaborn as sns

# Config
IMAGE_SIZE  = 128
BATCH_SIZE  = 32
EPOCHS      = 50
NUM_CLASSES = 10
DATASET_DIR = 'dataset/PlantVillage'

CLASS_NAMES = [
    'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight',
    'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites',
    'Tomato_Target_Spot', 'Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato_mosaic_virus', 'Tomato_healthy'
]

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## Step 4: Load & Preprocess Dataset

In [ ]:
def load_dataset(dataset_dir, class_names):
    images, labels = [], []
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    for cls in class_names:
        cls_dir = os.path.join(dataset_dir, cls)
        if not os.path.exists(cls_dir):
            print(f'WARNING: {cls_dir} not found, skipping.')
            continue
        files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        print(f'  {cls}: {len(files)} images')
        for fname in files:
            img = cv2.imread(os.path.join(cls_dir, fname))
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
            images.append(img)
            labels.append(class_to_idx[cls])
    return np.array(images, dtype='float32') / 255.0, np.array(labels, dtype='int32')

images, labels = load_dataset(DATASET_DIR, CLASS_NAMES)
print(f'\nTotal: {len(images)} images loaded')

# Split 70/15/15
X_tr, X_tmp, y_tr, y_tmp = train_test_split(images, labels, test_size=0.30, stratify=labels, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)
print(f'Train: {len(X_tr)} | Val: {len(X_val)} | Test: {len(X_te)}')

## Step 5: Data Augmentation

In [ ]:
y_tr_cat  = to_categorical(y_tr,  NUM_CLASSES)
y_val_cat = to_categorical(y_val, NUM_CLASSES)
y_te_cat  = to_categorical(y_te,  NUM_CLASSES)

train_gen = ImageDataGenerator(
    rotation_range=20, width_shift_range=0.15, height_shift_range=0.15,
    zoom_range=0.15, horizontal_flip=True, brightness_range=[0.8, 1.2]
).flow(X_tr, y_tr_cat, batch_size=BATCH_SIZE, shuffle=True)

val_gen = ImageDataGenerator().flow(X_val, y_val_cat, batch_size=BATCH_SIZE, shuffle=False)
print('Data generators ready.')

## Step 6: Build Custom CNN

In [ ]:
def build_cnn():
    m = models.Sequential([
        layers.Conv2D(32, 3, padding='same', activation='relu', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
        layers.BatchNormalization(), layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.25),

        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(), layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.25),

        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(), layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.30),

        layers.Conv2D(256, 3, padding='same', activation='relu'),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.30),

        layers.Flatten(),
        layers.Dense(512, activation='relu'), layers.BatchNormalization(), layers.Dropout(0.50),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
    return m

cnn = build_cnn()
cnn.summary()

## Step 7: Train the Model

In [ ]:
os.makedirs('models', exist_ok=True)

callbacks = [
    ModelCheckpoint('models/custom_cnn_best.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
]

history = cnn.fit(train_gen, epochs=EPOCHS, validation_data=val_gen, callbacks=callbacks, verbose=1)
print('Training complete!')

## Step 8: Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('models/training_curves.png', dpi=150); plt.show()

## Step 9: Evaluate on Test Set

In [ ]:
best_model = tf.keras.models.load_model('models/custom_cnn_best.h5')
y_pred_probs = best_model.predict(X_te, batch_size=32)
y_pred = np.argmax(y_pred_probs, axis=1)

from sklearn.metrics import accuracy_score
print(f'Test Accuracy: {accuracy_score(y_te, y_pred)*100:.2f}%')
print(classification_report(y_te, y_pred, target_names=CLASS_NAMES))

## Step 10: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_te, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
plt.figure(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix (Normalized)')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig('models/confusion_matrix.png', dpi=150); plt.show()

## Step 11: Save Class Names & Copy to Drive

In [ ]:
with open('models/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f)
print('class_names.json saved.')

# Copy models to Google Drive for persistence
import shutil
shutil.copytree('models', '/content/drive/MyDrive/TomatoDisease/models', dirs_exist_ok=True)
print('Models copied to Google Drive!')